In [16]:
import numpy as np
import pandas as pd
import time
import os
import gc
import copy

import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

print("GPU:", torch.cuda.get_device_name(0))
device = torch.device("cuda")

GPU: Tesla T4


In [17]:
TRAIN_PATH = "/content/relativistic_dataset_1m.npy"
EXTREME_PATH = "/content/relativistic_extreme_test_10k.npy"

data = np.load(TRAIN_PATH)
extreme_data = np.load(EXTREME_PATH)

print("Training dataset:", data.shape)
print("Extreme dataset :", extreme_data.shape)

Training dataset: (1000000, 13)
Extreme dataset : (10000, 13)


In [18]:
# Inputs
X = data[:, [0, 1, 2, 3, 4]]

# Outputs
Y = data[:, 5:13]

print("X:", X.shape)
print("Y:", Y.shape)

X: (1000000, 5)
Y: (1000000, 8)


In [19]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (800000, 5)
Testing : (200000, 5)


In [20]:
scaler_X = StandardScaler()
scaler_Y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

Y_train_scaled = scaler_Y.fit_transform(Y_train)
Y_test_scaled = scaler_Y.transform(Y_test)

print("Normalization complete.")

Normalization complete.


In [21]:
X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

Y_train_tensor = torch.tensor(
    Y_train_scaled,
    dtype=torch.float32
)

train_dataset = TensorDataset(
    X_train_tensor,
    Y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4096,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("Batches:", len(train_loader))

Batches: 196


In [22]:
class RelativisticMLP(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(5, 256),
            nn.ReLU(),

            nn.Linear(256, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 8)
        )

    def forward(self, x):

        return self.network(x)


model = RelativisticMLP().to(device)

print(model)

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

RelativisticMLP(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=8, bias=True)
  )
)
Parameters: 101256


In [23]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

EPOCHS = 50

best_loss = float("inf")
best_state = None

start_training = time.perf_counter()


for epoch in range(EPOCHS):

    model.train()

    total_loss = 0.0

    for batch_X, batch_Y in train_loader:

        batch_X = batch_X.to(
            device,
            non_blocking=True
        )

        batch_Y = batch_Y.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        prediction = model(batch_X)

        loss = criterion(
            prediction,
            batch_Y
        )

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item()
            * batch_X.size(0)
        )

    epoch_loss = (
        total_loss
        /
        len(train_dataset)
    )

    if epoch_loss < best_loss:

        best_loss = epoch_loss

        best_state = copy.deepcopy(
            model.state_dict()
        )

    print(
        f"Epoch {epoch+1:03d}/{EPOCHS}"
        f" | Loss = {epoch_loss:.8e}"
    )


training_time = (
    time.perf_counter()
    - start_training
)

print(
    "\nTraining time:",
    training_time,
    "seconds"
)

Epoch 001/50 | Loss = 5.16049581e-02
Epoch 002/50 | Loss = 1.04743578e-03
Epoch 003/50 | Loss = 4.67495439e-04
Epoch 004/50 | Loss = 4.02206541e-04
Epoch 005/50 | Loss = 3.01788706e-04
Epoch 006/50 | Loss = 2.55460780e-04
Epoch 007/50 | Loss = 2.42504250e-04
Epoch 008/50 | Loss = 2.50808936e-04
Epoch 009/50 | Loss = 1.88367673e-04
Epoch 010/50 | Loss = 1.82352516e-04
Epoch 011/50 | Loss = 1.77159129e-04
Epoch 012/50 | Loss = 1.85501511e-04
Epoch 013/50 | Loss = 2.58618533e-04
Epoch 014/50 | Loss = 1.03258115e-04
Epoch 015/50 | Loss = 1.52194427e-04
Epoch 016/50 | Loss = 1.51036168e-04
Epoch 017/50 | Loss = 1.55436797e-04
Epoch 018/50 | Loss = 1.58260171e-04
Epoch 019/50 | Loss = 1.33571345e-04
Epoch 020/50 | Loss = 1.40756823e-04
Epoch 021/50 | Loss = 1.40189335e-04
Epoch 022/50 | Loss = 1.61801519e-04
Epoch 023/50 | Loss = 1.16099201e-04
Epoch 024/50 | Loss = 1.09461652e-04
Epoch 025/50 | Loss = 1.46158514e-04
Epoch 026/50 | Loss = 1.22574682e-04
Epoch 027/50 | Loss = 1.41174265e-04
E

In [24]:
model.load_state_dict(best_state)

model.eval()

print("Best training loss:", best_loss)

Best training loss: 6.186447221785784e-05


In [25]:
X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32,
    device=device
)

start = time.perf_counter()

with torch.no_grad():

    prediction_scaled = model(
        X_test_tensor
    )

normal_prediction_time = (
    time.perf_counter()
    - start
)

prediction_scaled = (
    prediction_scaled
    .cpu()
    .numpy()
)

prediction_normal = (
    scaler_Y.inverse_transform(
        prediction_scaled
    )
)

print(
    "Prediction time:",
    normal_prediction_time,
    "sec"
)

Prediction time: 0.0013830660000166972 sec


In [26]:
target_names = [
    "position",
    "velocity",
    "beta",
    "gamma",
    "momentum",
    "kinetic_energy",
    "total_energy",
    "proper_time"
]


print("=" * 80)
print("NORMAL 200K TEST")
print("=" * 80)


for i, target in enumerate(target_names):

    actual = Y_test[:, i]
    predicted = prediction_normal[:, i]

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    mae = mean_absolute_error(
        actual,
        predicted
    )

    r2 = r2_score(
        actual,
        predicted
    )

    nrmse = (
        rmse / np.std(actual)
    )

    print(
        f"{target:20s}"
        f" | R² = {r2:.8f}"
        f" | NRMSE = {nrmse:.6e}"
    )

NORMAL 200K TEST
position             | R² = 0.99939428 | NRMSE = 2.461129e-02
velocity             | R² = 0.99946778 | NRMSE = 2.306981e-02
beta                 | R² = 0.99953215 | NRMSE = 2.162984e-02
gamma                | R² = 0.99894402 | NRMSE = 3.249589e-02
momentum             | R² = 0.99973007 | NRMSE = 1.642953e-02
kinetic_energy       | R² = 0.99957855 | NRMSE = 2.052919e-02
total_energy         | R² = 0.99996136 | NRMSE = 6.216072e-03
proper_time          | R² = 0.99997676 | NRMSE = 4.820498e-03


In [27]:
X_extreme = extreme_data[
    :,
    [0, 1, 2, 3, 4]
]

Y_extreme = extreme_data[
    :,
    5:13
]

X_extreme_scaled = scaler_X.transform(
    X_extreme
)

X_extreme_tensor = torch.tensor(
    X_extreme_scaled,
    dtype=torch.float32,
    device=device
)

print("Extreme X:", X_extreme.shape)
print("Extreme Y:", Y_extreme.shape)

Extreme X: (10000, 5)
Extreme Y: (10000, 8)


In [28]:
model.eval()

start = time.perf_counter()

with torch.no_grad():

    extreme_prediction_scaled = model(
        X_extreme_tensor
    )

extreme_prediction_time = (
    time.perf_counter()
    - start
)

extreme_prediction_scaled = (
    extreme_prediction_scaled
    .cpu()
    .numpy()
)

extreme_prediction = (
    scaler_Y.inverse_transform(
        extreme_prediction_scaled
    )
)

print(
    "Extreme prediction time:",
    extreme_prediction_time,
    "sec"
)

Extreme prediction time: 0.0022401930000341963 sec


In [29]:
nn_results = []

print("=" * 80)
print("NEURAL NETWORK — EXTREME 10K TEST")
print("=" * 80)


for i, target in enumerate(target_names):

    actual = Y_extreme[:, i]

    predicted = extreme_prediction[:, i]

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    mae = mean_absolute_error(
        actual,
        predicted
    )

    r2 = r2_score(
        actual,
        predicted
    )

    nrmse = (
        rmse / np.std(actual)
    )

    nonzero = (
        np.abs(actual)
        > np.finfo(float).eps
    )

    relative_error = np.mean(
        np.abs(
            predicted[nonzero]
            -
            actual[nonzero]
        )
        /
        np.abs(
            actual[nonzero]
        )
    )

    max_error = np.max(
        np.abs(
            predicted
            -
            actual
        )
    )


    print("\n" + "-" * 80)

    print(
        f"Target: {target}"
    )

    print(
        f"RMSE            : {rmse:.6e}"
    )

    print(
        f"MAE             : {mae:.6e}"
    )

    print(
        f"R²              : {r2:.8f}"
    )

    print(
        f"Normalized RMSE : {nrmse:.6e}"
    )

    print(
        f"Relative Error  : {relative_error:.6e}"
    )

    print(
        f"Max Error       : {max_error:.6e}"
    )


    nn_results.append({

        "model":
            "neural_network_mlp",

        "target":
            target,

        "rmse":
            rmse,

        "mae":
            mae,

        "r2":
            r2,

        "normalized_rmse":
            nrmse,

        "mean_relative_error":
            relative_error,

        "max_absolute_error":
            max_error,

        "prediction_time_sec":
            extreme_prediction_time

    })

NEURAL NETWORK — EXTREME 10K TEST

--------------------------------------------------------------------------------
Target: position
RMSE            : 2.922932e+12
MAE             : 6.769754e+11
R²              : 0.24482283
Normalized RMSE : 8.690093e-01
Relative Error  : 1.485306e+05
Max Error       : 4.137273e+13

--------------------------------------------------------------------------------
Target: velocity
RMSE            : 9.462454e+08
MAE             : 4.249768e+08
R²              : -55.34317915
Normalized RMSE : 7.506209e+00
Relative Error  : 6.554590e+00
Max Error       : 6.912303e+09

--------------------------------------------------------------------------------
Target: beta
RMSE            : 2.474562e+00
MAE             : 1.145748e+00
R²              : -33.63154159
Normalized RMSE : 5.884857e+00
Relative Error  : 8.615478e+00
Max Error       : 1.929645e+01

--------------------------------------------------------------------------------
Target: gamma
RMSE            : 1.7

In [30]:
nn_results_df = pd.DataFrame(
    nn_results
)

nn_results_df

,model,target,rmse,mae,r2,normalized_rmse,mean_relative_error,max_absolute_error,prediction_time_sec
0,neural_network_mlp,position,2.922932e+12,6.769754e+11,0.244823,0.869009,1.485306e+05,4.137273e+13,0.00224
1,neural_network_mlp,velocity,9.462454e+08,4.249768e+08,-55.343179,7.506209,6.554590e+00,6.912303e+09,0.00224
2,neural_network_mlp,beta,2.474562e+00,1.145748e+00,-33.631542,5.884857,8.615478e+00,1.929645e+01,0.00224
3,neural_network_mlp,gamma,1.792805e+04,5.819094e+02,-0.000449,1.000224,1.286148e+01,1.572290e+06,0.00224
4,neural_network_mlp,momentum,1.638362e+14,4.087584e+13,0.111576,0.942563,2.223588e+04,7.763232e+15,0.00224
5,neural_network_mlp,kinetic_energy,4.688811e+22,1.191798e+22,-0.010149,1.005062,3.392261e+07,2.280552e+24,0.00224
6,neural_network_mlp,total_energy,4.573905e+22,1.054065e+22,0.286693,0.844575,1.129384e+03,2.295473e+24,0.00224
7,neural_network_mlp,proper_time,5.549091e+03,1.632822e+03,0.668421,0.575829,1.743045e+05,5.932539e+04,0.00224
